# Ragrails — Embedding

This notebook covers `embed()` — reading chunk JSON files, generating vectors, and writing them to a vector database.

`store()` is kept as an alias for `embed()` if you prefer storage-oriented naming.

Supported providers:

- **Qdrant** — easiest local option
- **Pinecone** — managed serverless
- **Weaviate** — local or cloud

## Install

Pick the shortcut extra for your vector DB — each one bundles Voyage embeddings and the vector DB client:

```bash
pip install "ragrails[store-qdrant]"
pip install "ragrails[store-pinecone]"
pip install "ragrails[store-weaviate]"
```

Or install the providers separately:

```bash
pip install "ragrails[voyage,qdrant]"
pip install "ragrails[voyage,pinecone]"
pip install "ragrails[voyage,weaviate]"
```

In [ ]:
# Qdrant
# %pip install "ragrails[store-qdrant]"

# Pinecone
# %pip install "ragrails[store-pinecone]"

# Weaviate
# %pip install "ragrails[store-weaviate]"

## API keys

Set the Voyage API key before running `embed()`. Provider keys are only needed for Pinecone and Weaviate Cloud.

In [ ]:
import os

os.environ["VOYAGE_API_KEY"] = "your-voyage-api-key"

# Pinecone
# os.environ["PINECONE_API_KEY"] = "your-pinecone-api-key"

# Weaviate Cloud
# os.environ["WEAVIATE_API_KEY"] = "your-weaviate-api-key"

In [ ]:
from ragrails import RagRails

rag = RagRails()

---

## Qdrant

Qdrant is the simplest option for local development. Start it with Docker before running `embed()`.

In [ ]:
# Start Qdrant before running this cell:
# docker run -p 6333:6333 qdrant/qdrant

result = rag.embed(
    input_dir="files/output/chunks/web",
    vector_db="qdrant",
    url="http://localhost:6333",
    collection="rag_chunks",
)

print("Files embedded:", result.files)
print("Chunks embedded:", result.chunks)
print("Provider:", result.provider)
print("Collection:", result.collection)
print("Errors:", result.errors)

---

## Pinecone

Use hyphens in the collection name — Pinecone index names cannot contain underscores.

In [ ]:
result = rag.embed(
    input_dir="files/output/chunks/web",
    vector_db="pinecone",
    collection="rag-chunks",  # hyphens only, no underscores
)

print("Files embedded:", result.files)
print("Chunks embedded:", result.chunks)
print("Provider:", result.provider)
print("Collection:", result.collection)
print("Errors:", result.errors)

---

## Weaviate

Collection names must start with an uppercase letter and contain only letters and digits.

### Local Weaviate

In [ ]:
# Start Weaviate before running this cell (both HTTP and gRPC ports required):
# docker run -p 8080:8080 -p 50051:50051 cr.weaviate.io/semitechnologies/weaviate:1.36.9

result = rag.embed(
    input_dir="files/output/chunks/web",
    vector_db="weaviate",
    url="http://localhost:8080",
    collection="RagChunks",  # must start uppercase, letters and digits only
)

print("Files embedded:", result.files)
print("Chunks embedded:", result.chunks)
print("Provider:", result.provider)
print("Collection:", result.collection)
print("Errors:", result.errors)

### Weaviate Cloud

In [ ]:
result = rag.embed(
    input_dir="files/output/chunks/web",
    vector_db="weaviate",
    url="https://your-cluster.weaviate.cloud",
    collection="RagChunks",
)

print("Files embedded:", result.files)
print("Chunks embedded:", result.chunks)

---

## Embed from different ingestion sources

Point `input_dir` at any chunks folder created by `chunk()`.

In [ ]:
# Web chunks
rag.embed(input_dir="files/output/chunks/web", vector_db="qdrant", collection="rag_chunks")

# Document chunks
rag.embed(input_dir="files/output/chunks/docs", vector_db="qdrant", collection="rag_chunks")

# API chunks
rag.embed(input_dir="files/output/chunks/api", vector_db="qdrant", collection="rag_chunks")

## Embed selected files

Pass `files` to embed only specific chunk JSON files from the folder.

In [ ]:
result = rag.embed(
    input_dir="files/output/chunks/docs",
    files=["001_overview.json", "002_auth.json"],
    vector_db="qdrant",
    collection="rag_chunks",
)

print("Files embedded:", result.files)
print("Chunks embedded:", result.chunks)

## Custom embedding model

The default embedder is Voyage with `voyage-3`. Pass `model` to use a different Voyage model.

In [ ]:
result = rag.embed(
    input_dir="files/output/chunks/docs",
    vector_db="qdrant",
    collection="rag_chunks",
    embedder="voyage",
    model="voyage-3",
    batch_size=64,
)

print("Chunks embedded:", result.chunks)

---

## `store()` alias

`store()` accepts the same parameters as `embed()` and returns `StoreResult`. Use it if you prefer storage-oriented naming.

In [ ]:
result = rag.store(
    input_dir="files/output/chunks/docs",
    vector_db="qdrant",
    collection="rag_chunks",
)

print("Chunks stored:", result.chunks)

---

## What gets stored

Each vector point written to the database contains a stable ID, the dense vector, and a payload with the chunk text and metadata:

```python
{
    "id": "stable-chunk-id",
    "vector": [0.012, -0.034, ...],
    "payload": {
        "text": "chunk text",
        "source_url": "https://example.com/page",
        "title": "Page title",
        "content_hash": "...",
    }
}
```

Embedding is idempotent — re-running it for the same chunks updates existing vector points rather than creating duplicates. Chunk IDs are stable UUIDs derived from content.

---

## Error handling

In [ ]:
result = rag.embed(
    input_dir="files/output/chunks/web",
    vector_db="qdrant",
    collection="rag_chunks",
)

if result.errors:
    for error in result.errors:
        print("Error:", error)

### Common failure causes

| Provider | Common cause |
|---|---|
| All | `VOYAGE_API_KEY` missing or invalid |
| All | Input folder has no chunk JSON files |
| Qdrant | Qdrant not running, or port 6333 not exposed |
| Pinecone | `PINECONE_API_KEY` missing, or collection name has underscores |
| Weaviate | gRPC port 50051 not exposed, or collection name not uppercase |

---

## Next step

After embedding, run retrieval — see `04_retrieval.ipynb`.